# Small tanh-MLP mass model -- `tempest` run

This notebook tests `CHIMERA.population.mass.paired.nn_density` -- the
genuine (small) feedforward tanh-MLP paired mass model from `nn.py`, where
*every* weight and bias (`Ws`/`bs`) is a free, jointly-sampled hyperparameter
-- with a `tempest` hyperlikelihood run, mirroring
`examples/bpl3p_run_tempest.py` / `examples/random_features_blocks.ipynb` /
`examples/pls_free_knots_run_tempest.ipynb` (same priors-dict / `tempest` /
hyperlikelihood structure).

**Section 0** is fully local (no cluster data, no `tempest` needed) and
sanity-checks the plumbing: prior-predictive draws of the network, a direct
demonstration of why the prior on `Ws`/`bs` has to be calibrated (not flat),
and a jit/vmap check that sampling the whole network doesn't trigger
per-particle recompilation.

**Section 1** mirrors `bpl3p_run_tempest.py` for an actual `tempest`
hyperlikelihood run -- it needs the GWTC5 PE/injection files and the
`tempest` package (cluster-only), so edit `dir_data`/`dir_chain` before
running it.

## Why the prior needs to be calibrated, not flat

`nn.py`'s own docstring flags the danger: with every weight free to roam the
full posterior (rather than a handful of `feature_scales`/`spline_coefficients`),
a naive, flat prior on `Ws`/`bs` lets the network's output layer swing over
many orders of magnitude in `g(x) = log(p_tilde(m) * m)`, collapsing almost
all of the density into a random sliver of mass space. Every GW event then
fails its `pe_neff` effective-sample-size check and the hyperlikelihood is
`-inf` for essentially every prior draw -- `tempest` has no finite particles
to even seed its initial population from. Section 0.2 below demonstrates this
failure mode directly and shows the fix: a **fan-in-calibrated
("Xavier/LeCun-style") prior** on every hidden layer, plus an output-layer
prior scaled by `1/sqrt(hidden_sizes[-1])` to directly control `std(g)`. This
is the *same* calibration idea `random_features_blocks.ipynb` uses for
`w_out_prior_scale`, just applied per-layer instead of to a single vector.

`tempest` works with a flat parameter vector (`prior_transform`/`params_keys`),
so unlike a `numpyro` version of this notebook (which could sample each
weight *matrix* as one vector-valued site), every individual weight/bias
scalar gets its own flat-vector entry here -- `f"W{layer}_{row}_{col}"`,
`f"b{layer}_{row}"`, `f"Wout_{col}"` -- exactly the same style
`w_out{i}`/`spline_c{i}` already use in the other `tempest` notebooks,
just with two index levels instead of one.

In [ ]:
import os, sys
os.environ["CHIMERA_ENABLE_GPU"] = "True"
sys.path.append(os.getcwd() + '/../')

import numpy as np
import scipy.stats
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from CHIMERA.population.mass.paired import nn_density
from CHIMERA.population.mass import p_m1, p_m1m2


## Config

`hidden_sizes` is the MLP architecture (`nn.py` recommends keeping this
modest -- every weight is a sampled dimension). `hidden_prior_mult` and
`target_std_g` are the two knobs that replace `nn.py`'s default
uniform-init scheme with a *prior*:

- Each hidden layer's weight matrix gets an elementwise
  `Normal(0, hidden_prior_mult / sqrt(fan_in))` prior -- the classic
  Xavier/LeCun fan-in scaling, which keeps each layer's pre-activation
  variance roughly `O(hidden_prior_mult^2)` *regardless of layer width*, so
  widening a layer doesn't by itself make the network's output wilder.
- The bias-free output layer (mapping the last hidden layer -> the scalar
  log-density `g`) gets `Normal(0, target_std_g / sqrt(hidden_sizes[-1]))`:
  since the output is `g = W_out @ h` with `h = tanh(...)` bounded in
  `[-1, 1]`, this directly upper-bounds `std(g) ~ target_std_g` no matter how
  wide the last hidden layer is. `target_std_g = 1-2` keeps `g` (and hence
  `p_tilde(m) = exp(g)/m`) a mild, well-behaved modulation instead of an
  orders-of-magnitude swing.

`naive_scale` (Section 0.2 only) is a deliberately *uncalibrated* flat
`Normal(0, naive_scale)` prior applied to every weight/bias regardless of
fan-in, kept around purely to demonstrate why the calibration above matters.

In [ ]:
input_size = 1
hidden_sizes = (16, 16)          # keep modest -- every weight is a sampled dimension (see nn.py docstring)
depth = len(hidden_sizes)
sizes = (input_size,) + hidden_sizes
fan_ins = sizes[:-1]              # fan_ins[i] is the input width of hidden layer i

w_shapes = [(sizes[i + 1], sizes[i]) for i in range(depth)]
b_shapes = [(sizes[i + 1],) for i in range(depth)]
w_out_shape = (1, hidden_sizes[-1])

hidden_prior_mult = 1.5           # Xavier/LeCun-style multiplier on 1/sqrt(fan_in)
target_std_g = 1.5                # desired std of g(x) itself (the network's log-density output)
bias_prior_scale = 1.0            # biases don't need fan-in scaling (they add, not multiply, per unit)
naive_scale = 3.0                 # Section 0.2 only: deliberately *uncalibrated* flat prior, for comparison

hidden_scales = [float(hidden_prior_mult / np.sqrt(fi)) for fi in fan_ins]
out_scale = float(target_std_g / np.sqrt(hidden_sizes[-1]))

master_key = jax.random.PRNGKey(0)

# tempest / hyperlikelihood run settings (Section 1) -- modest defaults for a *test* run;
# bump these up for a production fit, same knobs as bpl3p_run_tempest.py's argparse flags.
n_particles = 256
n_steps = 5
n_total = 4096
prng_key = 0
resample = 'syst'

dir_data = '/leonardo_work/IscrC_MLGW/gwtc5/data/'
dir_chain = '/leonardo_work/IscrC_MLGW/gwtc5/res/tempest/'
chain_label = 'gwtc5_nn_density_tempest'

print("fan_ins       =", fan_ins)
print("hidden_scales =", hidden_scales)
print("out_scale     =", out_scale)


## Priors

Same style as `bpl3p_run_tempest.py`/`pls_free_knots_run_tempest.ipynb`:
`scipy.stats`-based `uniform`/`loguniform`/`norm` helpers, one prior per
scalar hyperparameter. Every weight/bias *element* gets its own flat-vector
entry (`f"W{i}_{r}_{c}"`, `f"b{i}_{r}"`, `f"Wout_{c}"`), later
re-assembled by `assemble_Ws_bs` into the `Ws`/`bs` lists
`nn_density.update()` expects -- exactly the pattern `w_out{i}`/`spline_c{i}`
already use in the other `tempest` notebooks, just with a 2D index.

In [ ]:
def uniform(low, high):
  return scipy.stats.uniform(loc=low, scale=high - low)

def norm(loc, scale):
  return scipy.stats.norm(loc=loc, scale=scale)

def w_key(i, r, c): return f"W{i}_{r}_{c}"
def b_key(i, r):    return f"b{i}_{r}"
def wout_key(c):    return f"Wout_{c}"

def assemble_Ws_bs(raw):
  # Pops every W{i}_{r}_{c}/b{i}_{r}/Wout_{c} entry out of `raw` (in place) and
  # returns the (Ws, bs) lists nn_density.update() expects. Whatever is left in
  # `raw` afterwards is the plain scalar hyperparams (beta, m_low, ...).
  Ws, bs = [], []
  for i in range(depth):
    W_i = jnp.array([[raw.pop(w_key(i, r, c)) for c in range(w_shapes[i][1])] for r in range(w_shapes[i][0])])
    b_i = jnp.array([raw.pop(b_key(i, r)) for r in range(b_shapes[i][0])])
    Ws.append(W_i)
    bs.append(b_i)
  W_out = jnp.array([[raw.pop(wout_key(c)) for c in range(hidden_sizes[-1])]])
  Ws.append(W_out)
  return Ws, bs

# --- mass-model (nn_density) priors ---
mass_priors = {
  "beta": uniform(-4., 12.),               # mass-ratio pairing power law
  "m_low": uniform(0.4, 1.4),
  "m_high": uniform(50., 200.),
}

# fan-in-calibrated priors, one per hidden-layer weight/bias element
layer_priors = {}
for i in range(depth):
  for r in range(w_shapes[i][0]):
    for c in range(w_shapes[i][1]):
      layer_priors[w_key(i, r, c)] = norm(0., hidden_scales[i])
  for r in range(b_shapes[i][0]):
    layer_priors[b_key(i, r)] = norm(0., bias_prior_scale)

# output-layer priors, calibrated by hidden_sizes[-1] to directly control std(g)
out_priors = {wout_key(c): norm(0., out_scale) for c in range(hidden_sizes[-1])}

# --- cosmology / rate priors (same as bpl3p_run_tempest.py) ---
cosmo_rate_priors = {
  "H0": uniform(10., 200.),
  "gamma": uniform(0., 12.),
  "kappa": uniform(0., 6.),
  "zp": uniform(0., 4.),
}

priors = {**cosmo_rate_priors, **mass_priors, **layer_priors, **out_priors}
params_keys = list(priors.keys())
n_dim = len(params_keys)
prior_dists = [priors[k] for k in params_keys]

def prior_transform(u):
  return np.array([dist.ppf(u[i]) for i, dist in enumerate(prior_dists)])

n_weight_params = sum(np.prod(s) for s in w_shapes) + int(np.prod(w_out_shape))
n_bias_params = sum(np.prod(s) for s in b_shapes)
print(f"n_dim = {n_dim}  ({n_weight_params} weights + {n_bias_params} biases + "
      f"{len(mass_priors)} mass + {len(cosmo_rate_priors)} cosmo/rate)")


## Section 0 -- local sanity checks (no cluster data / `tempest` needed)

### 0.1 Prior-predictive draws from the calibrated prior

Draw `Ws`/`bs` straight from the fan-in-calibrated priors above
(`scipy.stats.rvs`), build an `nn_density` realization for each draw, and
overlay `p(m1)`. Because the priors are calibrated (not arbitrarily wide),
each draw is a plausible, mildly-varying mass function -- this checks that
the model behaves sensibly as the *entire network* moves around its prior,
not just a handful of scale parameters.

In [ ]:
def draw_nn_weights(rng, hidden_scales_=hidden_scales, bias_scale_=bias_prior_scale, out_scale_=out_scale):
  # Draw one (Ws, bs) pair from the given per-layer scales (calibrated by default).
  Ws, bs = [], []
  for i in range(depth):
    Ws.append(jnp.array(scipy.stats.norm(0., hidden_scales_[i]).rvs(size=w_shapes[i], random_state=rng)))
    bs.append(jnp.array(scipy.stats.norm(0., bias_scale_).rvs(size=b_shapes[i], random_state=rng)))
  Ws.append(jnp.array(scipy.stats.norm(0., out_scale_).rvs(size=w_out_shape, random_state=rng)))
  return Ws, bs

n_draws = 12
m_low_fixed, m_high_fixed = 0.92, 124.2  # fixed mass range for this prior-predictive check

base_model = nn_density(
  input_size=input_size, hidden_sizes=hidden_sizes,
  m_low=m_low_fixed, m_high=m_high_fixed, key=master_key,
)

rng = np.random.default_rng(1)
m1 = jnp.linspace(m_low_fixed * 1.01, m_high_fixed * 0.99, 4000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for i in range(n_draws):
  Ws_i, bs_i = draw_nn_weights(rng)
  model_i = base_model.update(Ws=Ws_i, bs=bs_i)
  ax.plot(m1, p_m1(model_i, m1), alpha=0.6, lw=1, color=f"C{i % 10}")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$m_1$"); ax.set_ylabel("$p(m_1)$")
ax.set_title(f"{n_draws} calibrated-prior draws of the tanh-MLP mass function")
fig.tight_layout()
plt.show()


### 0.2 Why calibration matters: naive vs. fan-in-calibrated prior

Draw many (`Ws`, `bs`) samples from both the calibrated prior above and a
deliberately *uncalibrated* flat `Normal(0, naive_scale)` prior (same scale on
every weight/bias, ignoring fan-in and output width), and look at the
resulting mass function's dynamic range on a fixed grid,
`max(p(m1)) / min(p(m1))`. This is a local, cluster-independent proxy for
`nn.py`'s failure mode: a mass function with an astronomically large dynamic
range concentrates essentially all of its density in a single narrow spike,
which is exactly what makes every event's `pe_neff` check fail (`-inf`
hyperlikelihood) once real data is involved (Section 1) -- i.e. what "random
prior draws all get discarded" looks like in practice. The fraction of draws
kept below a generous `dynamic_range_threshold` is the local stand-in for
"fraction of prior draws with a finite log-likelihood" computed for real in
Section 1's prior-predictive check.

In [ ]:
def dynamic_range(model, Ws, bs, m_grid):
  model_i = model.update(Ws=Ws, bs=bs)
  p = p_m1(model_i, m_grid)
  p = jnp.where(jnp.isfinite(p), p, jnp.nan)
  p_pos = jnp.where(p > 0, p, jnp.nan)
  pmax, pmin = jnp.nanmax(p), jnp.nanmin(p_pos)
  ok = jnp.isfinite(pmax) & jnp.isfinite(pmin) & (pmax > 0)
  return jnp.where(ok, pmax / jnp.maximum(pmin, 1e-300), jnp.inf)

n_check_local = 2000
m1_check = jnp.linspace(m_low_fixed * 1.01, m_high_fixed * 0.99, 2000)
rng_c, rng_n = np.random.default_rng(2), np.random.default_rng(3)

dyn_calibrated, dyn_naive = [], []
for _ in range(n_check_local):
  Ws_c, bs_c = draw_nn_weights(rng_c)
  Ws_n, bs_n = draw_nn_weights(rng_n, [naive_scale] * depth, naive_scale, naive_scale)
  dyn_calibrated.append(float(dynamic_range(base_model, Ws_c, bs_c, m1_check)))
  dyn_naive.append(float(dynamic_range(base_model, Ws_n, bs_n, m1_check)))

dyn_calibrated = np.asarray(dyn_calibrated)
dyn_naive = np.asarray(dyn_naive)
dynamic_range_threshold = 1e4

frac_calibrated = np.mean(np.isfinite(dyn_calibrated) & (dyn_calibrated < dynamic_range_threshold))
frac_naive = np.mean(np.isfinite(dyn_naive) & (dyn_naive < dynamic_range_threshold))
print(f"calibrated prior: {frac_calibrated * 100:5.1f}% of {n_check_local} draws stay below "
      f"dynamic-range={dynamic_range_threshold:.0e}  (median dyn. range = {np.nanmedian(dyn_calibrated):.3g})")
print(f"naive prior:      {frac_naive * 100:5.1f}% of {n_check_local} draws stay below "
      f"dynamic-range={dynamic_range_threshold:.0e}  (median dyn. range = {np.nanmedian(dyn_naive):.3g})")

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
bins = np.logspace(0, 20, 60)
ax.hist(np.clip(dyn_naive, 1, 1e20), bins=bins, alpha=0.5, label='naive (flat) prior', color='C3')
ax.hist(np.clip(dyn_calibrated, 1, 1e20), bins=bins, alpha=0.7, label='fan-in-calibrated prior', color='C0')
ax.axvline(dynamic_range_threshold, color='k', ls='--', lw=1, label='threshold')
ax.set_xscale('log')
ax.set_xlabel('max(p(m1)) / min(p(m1))  (dynamic range per draw)')
ax.set_ylabel('count')
ax.legend()
fig.tight_layout()
plt.show()


### 0.3 jit/vmap check: one compile, many `Ws`/`bs` values

Batch several particles with *different* `Ws`/`bs` through `.update()` +
`p_m1` under a single `jax.jit` + `jax.vmap`, exactly the way `tempest` will
evaluate a particle population. The second call reuses the trace (no
recompilation) even though its `Ws`/`bs` values differ from the first call's.

In [ ]:
import time

n_particles_test = 64
Ws_bs_batch = [draw_nn_weights(rng) for _ in range(n_particles_test)]
Ws_batch = [jnp.stack([wb[0][i] for wb in Ws_bs_batch]) for i in range(depth + 1)]
bs_batch = [jnp.stack([wb[1][i] for wb in Ws_bs_batch]) for i in range(depth)]

@jax.jit
def eval_particle(Ws, bs):
  model_i = base_model.update(Ws=Ws, bs=bs)
  return p_m1(model_i, m1)

t0 = time.time()
out1 = jax.vmap(eval_particle)(Ws_batch, bs_batch)
out1.block_until_ready()
t1 = time.time()

bs_batch_2 = [b * 1.05 for b in bs_batch]  # different concrete values, same shapes
out2 = jax.vmap(eval_particle)(Ws_batch, bs_batch_2)
out2.block_until_ready()
t2 = time.time()

print(f"first call (compile + run): {t1 - t0:.3f}s")
print(f"second call, different bs values (should be much faster -- no retrace): {t2 - t1:.3f}s")
assert not jnp.any(jnp.isnan(out1)) and not jnp.any(jnp.isnan(out2))
print("no NaNs -- vmap over distinct Ws/bs works with a single jit trace.")


## Section 1 -- full `tempest` hyperlikelihood run (cluster only)

Mirrors `examples/bpl3p_run_tempest.py` / `examples/random_features_blocks.ipynb`
exactly in structure: load GWTC5 PE + injection data, build
`cosmo`/`mass`/`rate` -> `population` -> `hyperlikelihood`, define
`log_likelihood`, run `tempest.Sampler`, save posterior + blobs to
`arviz.InferenceData`.

Needs `dir_data`/`dir_chain` (set in the config cell above) to point at real
GWTC5 files, and the `tempest` package (both cluster-only) -- **edit the
paths before running this section**.

In [ ]:
from CHIMERA import data
from CHIMERA.cosmo import flrw
from CHIMERA.rate import madau_dickinson
from CHIMERA import population as population_module, selection_function, hyperlikelihood
import h5py
import tempest as tp

## Load PE data
file_ev = dir_data + 'GWTC5_242CBC_FAR0.25_PE2048.h5'
pe_gw = data.load_gw_pe_samples(file_ev, parameters=['m1det', 'm2det', 'dL'], return_struct=True)
pe_prior = jnp.load(dir_data + 'GWTC5_242CBC_FAR0.25_PE2048prior.npy')
pe_gw = pe_gw.update(pe_prior=pe_prior)
Nev = len(pe_gw.dL)

## Load injection data and define selection function
file_inj = dir_data + "GWTC5_injections_far0.25.h5"
f_inj = h5py.File(file_inj)
Ngen = float(f_inj.attrs['Ngen'])
Tobs = float(f_inj.attrs['Tobs'])
theta_inj_det = data.load_injection_data(file_inj, snr_cut=None, key_mapping={'snr': 'snr', 'log_pdraw': 'log_pdraw'}, frame='detector')

sel_fcn = selection_function(theta_inj_det, N_inj=Ngen)


In [ ]:
cosmo = flrw(H0=67.9, Om0=0.3065, z_max=5.)
mass = nn_density(
  input_size=input_size, hidden_sizes=hidden_sizes,
  m_low=0.9, m_high=124., key=master_key,
)
rate = madau_dickinson()
population = population_module(cosmo, mass, rate, scale_free=True, Tobs=Tobs, R0=1)

hyperlike = hyperlikelihood(
  theta_gw_det=pe_gw,
  population=population,
  z_grids_res=300,
  selection_function=sel_fcn,
  pe_neff=2.0,
  inj_neff=None,
  kind_kde='fft',
  kernel='epan',
  kde_bw=None,
)


Assembling `hyperparams` from the flat `tempest` particle vector:
`dict(zip(params_keys, params))` gives every scalar hyperparameter directly;
`assemble_Ws_bs` (defined in the Priors cell) pops out every
`W{i}_{r}_{c}`/`b{i}_{r}`/`Wout_{c}` entry and re-stacks them into the
`Ws`/`bs` lists that `nn_density.update()` (via `mass.keys`) expects --
exactly the same pattern `random_features_blocks.ipynb` uses for `w_out{i}`,
just with the 2D index handled inside `assemble_Ws_bs`.

In [ ]:
def log_likelihood(params):
  raw = dict(zip(params_keys, params))
  Ws, bs = assemble_Ws_bs(raw)
  hyperparams = {**raw, "Ws": Ws, "bs": bs}
  log_like_evs, N_exp, neff_inj, log_like = hyperlike.compute_all(**hyperparams)
  return log_like, log_like_evs, N_exp, neff_inj

# Prior-predictive finite-likelihood check before launching the full sampler: with n_particles
# particles drawn from the prior, tempest needs enough of them finite to seed its initial
# population from. Keep the accepted draws around too, to inspect what they look like as p(m1).
n_check = 10000
accepted_thetas = []
for s in range(n_check):
  theta = prior_transform(np.random.default_rng(s).uniform(size=n_dim))
  ll, *_ = log_likelihood(theta)
  if np.isfinite(ll):
    accepted_thetas.append(theta)
print(f"{len(accepted_thetas)}/{n_check} prior draws gave a finite log-likelihood "
      f"(~{len(accepted_thetas)/n_check*n_particles:.0f} expected finite particles at init)")


### What does the finite-likelihood region of the prior look like as a mass function?

Before spending compute on the full run, plot `p(m1)` for a handful of the
`accepted_thetas` collected above -- these are draws that passed every
event's `pe_neff` check, so this is a sanity check on what the sampler's
*initial* population actually looks like, not yet the posterior.

In [ ]:
n_plot = min(20, len(accepted_thetas))
plot_idx = np.random.default_rng(0).choice(len(accepted_thetas), size=n_plot, replace=False)
m1_grid = jnp.linspace(0.5, 200., 3000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for i in plot_idx:
  raw = dict(zip(params_keys, accepted_thetas[i]))
  Ws_i, bs_i = assemble_Ws_bs(raw)
  mass_i = mass.update(
    Ws=Ws_i, bs=bs_i,
    beta=raw["beta"], m_low=raw["m_low"], m_high=raw["m_high"],
  )
  ax.plot(m1_grid, p_m1(mass_i, m1_grid), alpha=0.6, lw=1)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$m_1$"); ax.set_ylabel("$p(m_1)$")
ax.set_title(f"{n_plot} accepted (finite-likelihood) prior draws")
fig.tight_layout()
plt.show()


In [ ]:
sampler = tp.Sampler(
  prior_transform=prior_transform,
  log_likelihood=log_likelihood,
  n_dim=n_dim,
  n_particles=n_particles,
  n_steps=n_steps,
  resample=resample,
  random_state=prng_key,
  output_dir=dir_chain,
  output_label=chain_label,
)

sampler.run(n_total=n_total, save_every=1, resume_state_path=None)

samples, weights, logl, blobs = sampler.posterior(resample=True, return_blobs=True)
samples_dict = {k: samples[:, i] for i, k in enumerate(priors)}

log_like_evs_blob = blobs[:, 0]
N_exp_blob = blobs[:, 1]
neff_inj_blob = blobs[:, 2]

logZ, logZerr = sampler.evidence()
print(f"logZ = {logZ}  logZerr = {logZerr}")


In [ ]:
import xarray as xr
import arviz as az

n_chains = 1
n_draws_post = len(samples)
n_events = np.stack(log_like_evs_blob).shape[-1]

posterior_ds = xr.Dataset(
  {k: (['chain', 'draw'], v.reshape(n_chains, n_draws_post)) for k, v in samples_dict.items()}
)
sample_stats_ds = xr.Dataset({
  'log_likelihood': (['chain', 'draw'], logl.reshape(n_chains, n_draws_post)),
  'weights': (['chain', 'draw'], weights.reshape(n_chains, n_draws_post)),
  'log_like_evs': (['chain', 'draw', 'event'], np.stack(log_like_evs_blob).reshape(n_chains, n_draws_post, n_events)),
  'N_exp': (['chain', 'draw'], np.asarray(N_exp_blob).reshape(n_chains, n_draws_post)),
  'Neff_inj': (['chain', 'draw'], np.asarray(neff_inj_blob).reshape(n_chains, n_draws_post)),
})

idata = az.InferenceData(posterior=posterior_ds, sample_stats=sample_stats_ds)
idata.attrs['log_evidence'] = float(logZ)
idata.attrs['log_evidence_err'] = str(logZerr) if logZerr is None else float(logZerr)
idata.to_netcdf(f"{dir_chain}/{chain_label}_final.nc")


## Section 2 -- inspecting the network in the posterior

Compares the posterior over the output-layer weights (`Wout_{c}`, the
hyperparameters most directly responsible for `g`'s overall scale) against
their calibrated prior -- if the posterior tightens noticeably, the data is
informative about the network's output scale. Also draws a
posterior-predictive `p(m1)` band using the sampled `Ws`/`bs`/mass-edge
hyperparameters together.

In [ ]:
wout_keys = [wout_key(c) for c in range(hidden_sizes[-1])]
wout_post = np.concatenate([samples_dict[k] for k in wout_keys])
wout_prior_draws = np.concatenate([
  out_priors[k].rvs(size=5000, random_state=np.random.default_rng(300 + i)) for i, k in enumerate(wout_keys)
])

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(wout_prior_draws, bins=40, density=True, alpha=0.4, label='prior', color='gray')
ax.hist(wout_post, bins=40, density=True, alpha=0.7, label='posterior', color='C0')
ax.set_title("output-layer weights $W_{\\rm out}$ (pooled entries)")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
n_ppc = 200
idx = np.random.default_rng(0).choice(len(samples), size=n_ppc, replace=False)
m1_grid = jnp.linspace(0.5, 150., 3000)

curves = []
for i in idx:
  raw = {k: samples_dict[k][i] for k in params_keys}
  Ws_i, bs_i = assemble_Ws_bs(raw)
  model_i = mass.update(
    Ws=Ws_i, bs=bs_i,
    beta=raw["beta"], m_low=raw["m_low"], m_high=raw["m_high"],
  )
  curves.append(p_m1(model_i, m1_grid))

curves = jnp.stack(curves)
lo, med, hi = jnp.percentile(curves, jnp.array([5., 50., 95.]), axis=0)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.fill_between(m1_grid, lo, hi, alpha=0.3, label='90% posterior-predictive')
ax.plot(m1_grid, med, lw=1.5, label='median')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.legend()
fig.tight_layout()
plt.show()
